# 08 - Ön Detection Parametre Kalibrasyonu

Bu notebook, valid görüntülerden seçilen küçük bir alt küme üzerinde sliding-window detection parametrelerini kalibre eder.

Amaç model eğitmek değil; Stage 07'den gelen sınıflandırma modellerinin detection aşamasında kullanacağı eşik, stride ve kümeleme ayarlarını belirlemektir.


## 1. Kütüphaneler

Bu bölümde dosya işlemleri, tablo üretimi, görüntü işleme, model yükleme ve görselleştirme için kullanılan kütüphaneler yüklenir.


In [ ]:
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from itertools import product
import time
import json
import re
import warnings

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.feature import hog, local_binary_pattern

warnings.filterwarnings("ignore")

print("Kütüphaneler yüklendi.")


## 2. Ayarlar

Bu bölümde valid alt küme seçimi, detection parametre aralıkları ve overwrite davranışı tanımlanır.


In [ ]:
RANDOM_STATE = 42

OVERWRITE_TABLES = False
OVERWRITE_FIGURES = False

USE_VALID_SUBSET = True
RUN_PRELIMINARY_CALIBRATION = True
PRELIMINARY_CALIBRATION_MODE = "on_the_fly"

IMAGE_SCALE = 1.0
POSTPROCESS_METHOD = "weighted_center"
MAX_CANDIDATES_PER_IMAGE = 500

MATCHING_IOU_THRESHOLDS = [0.10, 0.30]
PRIMARY_MATCHING_IOU = 0.30

VALID_SUBSET_TARGETS = {
    0: 20,
    1: 20,
    2: 20,
    3: 10,
}

LINEAR_PARAM_GRID = {
    "score_threshold": [1.0, 1.5, 2.0],
    "stride_divisor": [4, 6],
    "cluster_iou_threshold": [0.20, 0.30],
    "weighted_box_size_factor": [1.00, 1.25],
}

RBF_PARAM_GRID = {
    "decision_function_score_threshold": [0.0, 0.25, 0.50, 0.75],
    "stride_divisor": [4, 6],
    "cluster_iou_threshold": [0.20, 0.30],
    "weighted_box_size_factor": [1.00, 1.25],
}

RF_PARAM_GRID = {
    "predict_proba_score_threshold": [0.30, 0.40, 0.50, 0.60],
    "stride_divisor": [4, 6],
    "cluster_iou_threshold": [0.20, 0.30],
    "weighted_box_size_factor": [1.00, 1.25],
}

PARAM_GRIDS = {
    "linear_svm": LINEAR_PARAM_GRID,
    "rbf_svm": RBF_PARAM_GRID,
    "random_forest": RF_PARAM_GRID,
}

THRESHOLD_COLUMN_BY_ALGORITHM = {
    "linear_svm": "score_threshold",
    "rbf_svm": "decision_function_score_threshold",
    "random_forest": "predict_proba_score_threshold",
}

ALGORITHM_DISPLAY_ORDER = ["linear_svm", "rbf_svm", "random_forest"]

print("Ayarlar yüklendi.")
print("Valid subset targets:", VALID_SUBSET_TARGETS)
print("Matching IoU thresholds:", MATCHING_IOU_THRESHOLDS)


## 3. Dosya Yolları

Bu bölümde proje kökü, 08 çıktı klasörü, önceki aşamadan gelen sıralama tabloları ve model havuzu yolları tanımlanır.


In [ ]:
def auto_find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()
    start_path = Path(start_path).resolve()
    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        has_repo_layout = (candidate / "approaches" / "traditional_ml").exists()
        has_data_dir = (candidate / "data").exists()
        if has_repo_layout and has_data_dir:
            return candidate
    return None


PROJECT_ROOT = auto_find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Proje kökü bulunamadı. Notebook'u repo içinde çalıştırın.")

APPROACH_ROOT = PROJECT_ROOT / "approaches" / "traditional_ml"
STAGE_NAME = "08_detection_screening"
NOTEBOOK_NAME = "00_preliminary_detection_parameter_calibration"

STAGE_OUTPUT_DIR = APPROACH_ROOT / "outputs" / STAGE_NAME / NOTEBOOK_NAME
TABLES_DIR = STAGE_OUTPUT_DIR / "tables"
FIGURES_DIR = STAGE_OUTPUT_DIR / "figures"

for directory in [TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DATASET_CONFIGS = {
    "dataset1_varroa": {
        "dataset_short": "dataset1",
        "dataset_index": 1,
        "varroa_class_ids": {1},
        "ranked_stage07_path": APPROACH_ROOT / "outputs" / "07_classification_final_test" / "01_dataset1_classification_final_test" / "tables" / "ranked_classification_final_test_models.csv",
        "raw_dataset_dir": PROJECT_ROOT / "data" / "raw" / "dataset1_varroa",
        "pca_artifacts_dir": PROJECT_ROOT / "data" / "features" / "dataset1_varroa" / "pca_artifacts",
        "stage03_feature_dir": PROJECT_ROOT / "data" / "features" / "dataset1_varroa",
        "model_pool_dir": PROJECT_ROOT / "outputs" / "models" / "selected_classification_model_pool" / "dataset1_varroa",
    },
    "dataset2_honeybee": {
        "dataset_short": "dataset2",
        "dataset_index": 2,
        "varroa_class_ids": {0},
        "ranked_stage07_path": APPROACH_ROOT / "outputs" / "07_classification_final_test" / "02_dataset2_classification_final_test" / "tables" / "ranked_classification_final_test_models.csv",
        "raw_dataset_dir": PROJECT_ROOT / "data" / "raw" / "dataset2_honeybee",
        "pca_artifacts_dir": PROJECT_ROOT / "data" / "features" / "dataset2_honeybee" / "pca_artifacts",
        "stage03_feature_dir": PROJECT_ROOT / "data" / "features" / "dataset2_honeybee",
        "model_pool_dir": PROJECT_ROOT / "outputs" / "models" / "selected_classification_model_pool" / "dataset2_honeybee",
    },
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("STAGE_OUTPUT_DIR:", STAGE_OUTPUT_DIR)


## 4. Kayıt ve Çıktı Yardımcıları

Bu bölümde tablo ve figür çıktıları için ortak kayıt fonksiyonları tanımlanır.


In [ ]:
output_log = []


def timestamp_now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def relative_path(path):
    path = Path(path)

    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def log_output(message):
    message = str(message)
    print(message)
    output_log.append({
        "type": "text",
        "content": message,
        "timestamp": timestamp_now(),
    })


def log_dataframe(title, df, max_rows=20):
    print(f"\n{title}")
    display(df.head(max_rows))


def save_csv(df, path, overwrite=False, note=""):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists() and not overwrite:
        log_output(f"[SKIP] Mevcut CSV korundu: {relative_path(path)}")
        try:
            return pd.read_csv(path)
        except pd.errors.EmptyDataError:
            log_output(f"[INFO] Mevcut CSV boş olduğu için güncel tablo bellekte kullanılacak: {relative_path(path)}")
            return df

    df.to_csv(path, index=False, encoding="utf-8-sig")
    log_output(f"[SAVE] CSV kaydedildi: {relative_path(path)} | shape={df.shape}")
    return df


def save_current_figure(path, title, overwrite=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists() and not overwrite:
        log_output(f"[SKIP] Mevcut figür korundu: {relative_path(path)}")
        return path

    plt.savefig(path, dpi=150, bbox_inches="tight")
    log_output(f"[SAVE] Figür kaydedildi: {relative_path(path)} | {title}")
    return path


print("Yardımcı fonksiyonlar hazır.")


## 5. Girdi Envanteri

Bu bölümde kullanılacak ham veri klasörleri, Stage 07 sıralama tabloları, feature yardımcı dosyaları ve model havuzu kontrol edilir.


In [ ]:
input_inventory_rows = []

def add_inventory_item(name, path, item_type):
    path = Path(path)
    input_inventory_rows.append({
        "name": name,
        "path": relative_path(path),
        "item_type": item_type,
        "exists": path.exists(),
    })

for dataset_name, cfg in DATASET_CONFIGS.items():
    add_inventory_item(f"{dataset_name}_ranked_stage07", cfg["ranked_stage07_path"], "file")
    add_inventory_item(f"{dataset_name}_raw_dataset_dir", cfg["raw_dataset_dir"], "dir")
    add_inventory_item(f"{dataset_name}_valid_images", cfg["raw_dataset_dir"] / "valid" / "images", "dir")
    add_inventory_item(f"{dataset_name}_valid_labels", cfg["raw_dataset_dir"] / "valid" / "labels", "dir")
    add_inventory_item(f"{dataset_name}_model_pool_dir", cfg["model_pool_dir"], "dir")
    for algorithm_name in ALGORITHM_DISPLAY_ORDER:
        add_inventory_item(f"{dataset_name}_{algorithm_name}_model_dir", cfg["model_pool_dir"] / algorithm_name, "dir")
    add_inventory_item(f"{dataset_name}_pca_artifacts_dir", cfg["pca_artifacts_dir"], "dir")
    add_inventory_item(f"{dataset_name}_stage03_feature_dir", cfg["stage03_feature_dir"], "dir")

input_inventory_df = pd.DataFrame(input_inventory_rows)
save_csv(input_inventory_df, TABLES_DIR / "input_inventory.csv", overwrite=OVERWRITE_TABLES)
log_dataframe("Input Inventory", input_inventory_df)

missing_inputs_df = input_inventory_df[~input_inventory_df["exists"]].copy()
if len(missing_inputs_df) > 0:
    log_dataframe("Missing Inputs", missing_inputs_df)
    raise FileNotFoundError("Some required input files/folders are missing. See input_inventory.csv.")

log_output("[OK] All required input files/folders exist.")


## 6. YOLO Etiket Okuma Yardımcıları

Bu bölümde valid split içindeki görüntü ve YOLO etiket dosyaları okunarak görüntü başına Varroa sayısı çıkarılır.


In [ ]:
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

def find_image_files(images_dir):
    images_dir = Path(images_dir)
    image_paths = []
    for ext in IMAGE_EXTENSIONS:
        image_paths.extend(images_dir.glob(f"*{ext}"))
        image_paths.extend(images_dir.glob(f"*{ext.upper()}"))
    return sorted(set(image_paths))

def parse_yolo_label_file(label_path, image_width, image_height, varroa_class_ids):
    label_path = Path(label_path)
    valid_boxes = []
    invalid_rows = []

    if not label_path.exists():
        return valid_boxes, invalid_rows, True

    text = label_path.read_text(encoding="utf-8", errors="ignore").strip()
    if text == "":
        return valid_boxes, invalid_rows, False

    for row_index, line in enumerate(text.splitlines()):
        raw_line = line.strip()
        if raw_line == "":
            continue

        parts = raw_line.split()
        row_error = None

        if len(parts) != 5:
            row_error = "wrong_column_count"
        else:
            try:
                class_id = int(float(parts[0]))
                x_center_norm = float(parts[1])
                y_center_norm = float(parts[2])
                width_norm = float(parts[3])
                height_norm = float(parts[4])

                values = [x_center_norm, y_center_norm, width_norm, height_norm]
                if not all(np.isfinite(values)):
                    row_error = "non_finite_value"
                elif not (0 <= x_center_norm <= 1 and 0 <= y_center_norm <= 1):
                    row_error = "center_out_of_range"
                elif not (0 < width_norm <= 1 and 0 < height_norm <= 1):
                    row_error = "size_out_of_range"
                else:
                    x_center_px = x_center_norm * image_width
                    y_center_px = y_center_norm * image_height
                    width_px = width_norm * image_width
                    height_px = height_norm * image_height

                    x1 = x_center_px - width_px / 2
                    y1 = y_center_px - height_px / 2
                    x2 = x_center_px + width_px / 2
                    y2 = y_center_px + height_px / 2

                    box = {
                        "row_index": int(row_index),
                        "class_id": int(class_id),
                        "x_center_norm": float(x_center_norm),
                        "y_center_norm": float(y_center_norm),
                        "width_norm": float(width_norm),
                        "height_norm": float(height_norm),
                        "x1": float(max(0, min(image_width, x1))),
                        "y1": float(max(0, min(image_height, y1))),
                        "x2": float(max(0, min(image_width, x2))),
                        "y2": float(max(0, min(image_height, y2))),
                        "is_varroa": bool(class_id in varroa_class_ids),
                    }
                    valid_boxes.append(box)
            except Exception as exc:
                row_error = f"parse_error:{type(exc).__name__}"

        if row_error is not None:
            invalid_rows.append({
                "row_index": int(row_index),
                "raw_line": raw_line,
                "error": row_error,
            })

    return valid_boxes, invalid_rows, False

def build_valid_image_annotation_summary(dataset_name, cfg):
    images_dir = cfg["raw_dataset_dir"] / "valid" / "images"
    labels_dir = cfg["raw_dataset_dir"] / "valid" / "labels"
    varroa_class_ids = cfg["varroa_class_ids"]

    rows = []
    image_paths = find_image_files(images_dir)

    for image_path in image_paths:
        image = cv2.imread(str(image_path))
        label_path = labels_dir / f"{image_path.stem}.txt"

        if image is None:
            rows.append({
                "dataset_name": dataset_name,
                "split": "valid",
                "image_id": image_path.stem,
                "image_name": image_path.name,
                "image_path": relative_path(image_path),
                "label_path": relative_path(label_path),
                "image_width": np.nan,
                "image_height": np.nan,
                "readable_image": False,
                "empty_label_file": False,
                "valid_yolo_row_count": 0,
                "invalid_yolo_row_count": 0,
                "varroa_count": 0,
                "has_invalid_yolo_row": False,
                "eligible_for_subset": False,
                "gt_boxes_json": "[]",
            })
            continue

        image_height, image_width = image.shape[:2]
        valid_boxes, invalid_rows, empty_label_file = parse_yolo_label_file(
            label_path=label_path,
            image_width=image_width,
            image_height=image_height,
            varroa_class_ids=varroa_class_ids,
        )

        varroa_boxes = [box for box in valid_boxes if box["is_varroa"]]
        has_invalid = len(invalid_rows) > 0

        rows.append({
            "dataset_name": dataset_name,
            "split": "valid",
            "image_id": image_path.stem,
            "image_name": image_path.name,
            "image_path": relative_path(image_path),
            "label_path": relative_path(label_path),
            "image_width": int(image_width),
            "image_height": int(image_height),
            "readable_image": True,
            "empty_label_file": bool(empty_label_file),
            "valid_yolo_row_count": len(valid_boxes),
            "invalid_yolo_row_count": len(invalid_rows),
            "varroa_count": len(varroa_boxes),
            "has_invalid_yolo_row": has_invalid,
            "eligible_for_subset": (not has_invalid),
            "gt_boxes_json": json.dumps(varroa_boxes),
        })

    df = pd.DataFrame(rows)
    df["subset_group"] = df["varroa_count"].apply(
        lambda x: f"{int(x)}_varroa" if int(x) <= 3 else "4plus_varroa"
    )
    return df

print("YOLO parsing helpers ready.")


## 7. Valid Görüntü Özetleri

Bu bölümde iki veri setinin valid split görüntüleri için annotasyon özet tabloları hazırlanır.


In [ ]:
valid_summary_by_dataset = {}

for dataset_name, cfg in DATASET_CONFIGS.items():
    log_output("=" * 100)
    log_output(f"Building valid image annotation summary: {dataset_name}")

    summary_df = build_valid_image_annotation_summary(dataset_name, cfg)
    valid_summary_by_dataset[dataset_name] = summary_df

    output_name = f"valid_image_annotation_summary_{cfg['dataset_short']}.csv"
    save_csv(summary_df, TABLES_DIR / output_name, overwrite=OVERWRITE_TABLES)

    distribution_df = (
        summary_df
        .groupby(["varroa_count", "eligible_for_subset"], dropna=False)
        .size()
        .reset_index(name="image_count")
        .sort_values(["varroa_count", "eligible_for_subset"])
    )
    log_dataframe(f"{dataset_name} valid Varroa count distribution", distribution_df)

log_output("[OK] Valid image annotation summaries created.")


## 8. Valid Alt Küme Seçimi

Bu bölümde kalibrasyonu kısa tutmak için Varroa sayısına göre dengeli bir valid alt küme seçilir.


In [ ]:
def select_valid_subset(summary_df, targets, random_state=42):
    selected_parts = []
    selection_notes = []

    eligible_df = summary_df[
        (summary_df["readable_image"] == True) &
        (summary_df["eligible_for_subset"] == True)
    ].copy()

    for varroa_count, target_count in targets.items():
        group_df = eligible_df[eligible_df["varroa_count"] == varroa_count].copy()
        available_count = len(group_df)
        sample_count = min(target_count, available_count)

        if sample_count < target_count:
            note = (
                f"Requested {target_count} images for varroa_count={varroa_count}, "
                f"but only {available_count} eligible images are available. Taking all available."
            )
            selection_notes.append(note)
            log_output("[WARNING] " + note)

        if sample_count > 0:
            sampled_df = group_df.sample(
                n=sample_count,
                random_state=random_state + int(varroa_count),
                replace=False,
            ).copy()
            sampled_df["subset_target_count"] = target_count
            sampled_df["subset_available_count"] = available_count
            sampled_df["subset_sample_count"] = sample_count
            selected_parts.append(sampled_df)

    if len(selected_parts) == 0:
        return pd.DataFrame(), selection_notes

    selected_df = pd.concat(selected_parts, ignore_index=True)
    selected_df = selected_df.sort_values(["varroa_count", "image_name"]).reset_index(drop=True)
    selected_df["subset_order"] = np.arange(1, len(selected_df) + 1)
    selected_df["selection_random_state"] = random_state

    return selected_df, selection_notes

valid_subset_by_dataset = {}
subset_notes = []

for dataset_name, cfg in DATASET_CONFIGS.items():
    log_output("=" * 100)
    log_output(f"Selecting valid subset: {dataset_name}")

    selected_df, notes = select_valid_subset(
        valid_summary_by_dataset[dataset_name],
        VALID_SUBSET_TARGETS,
        RANDOM_STATE,
    )
    valid_subset_by_dataset[dataset_name] = selected_df
    subset_notes.extend([{"dataset_name": dataset_name, "note": note} for note in notes])

    output_name = f"valid_subset_selection_{cfg['dataset_short']}.csv"
    save_csv(selected_df, TABLES_DIR / output_name, overwrite=OVERWRITE_TABLES)

    subset_distribution_df = (
        selected_df
        .groupby("varroa_count")
        .size()
        .reset_index(name="selected_image_count")
        .sort_values("varroa_count")
    )
    log_dataframe(f"{dataset_name} selected valid subset distribution", subset_distribution_df)
    log_output(f"{dataset_name} selected subset image count: {len(selected_df)}")

if len(subset_notes) > 0:
    subset_notes_df = pd.DataFrame(subset_notes)
else:
    subset_notes_df = pd.DataFrame(columns=["dataset_name", "note"])

save_csv(subset_notes_df, TABLES_DIR / "valid_subset_selection_notes.csv", overwrite=OVERWRITE_TABLES)
log_output("[OK] Valid subsets selected.")


## 9. Anchor Model Seçimi

Bu bölümde Stage 07 final test sıralamalarından her veri seti ve algoritma için en iyi anchor model seçilir.


In [ ]:
def load_stage07_ranked_results(dataset_name, cfg):
    ranked_path = cfg["ranked_stage07_path"]
    df = pd.read_csv(ranked_path)
    df["dataset_name"] = dataset_name
    return df

def infer_patch_size_from_patch_set(patch_set):
    match = re.search(r"centered(\d+)", str(patch_set))
    if match is not None:
        return int(match.group(1))
    fallback = re.search(r"(\d+)", str(patch_set))
    if fallback is not None:
        return int(fallback.group(1))
    raise ValueError(f"Could not infer patch size from patch_set={patch_set}")

def find_model_artifact_path(dataset_name, algorithm_name, model_file_name):
    model_dir = DATASET_CONFIGS[dataset_name]["model_pool_dir"] / algorithm_name
    direct_path = model_dir / model_file_name
    if direct_path.exists():
        return direct_path

    matches = list(model_dir.rglob(model_file_name))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise FileExistsError(f"Multiple model artifact matches found for {model_file_name}: {matches}")
    raise FileNotFoundError(f"Model artifact not found: {model_file_name} under {model_dir}")

stage07_ranked_by_dataset = {}
anchor_rows = []

for dataset_name, cfg in DATASET_CONFIGS.items():
    ranked_df = load_stage07_ranked_results(dataset_name, cfg)
    stage07_ranked_by_dataset[dataset_name] = ranked_df

    for algorithm_name in ALGORITHM_DISPLAY_ORDER:
        group_df = ranked_df[ranked_df["algorithm_name"] == algorithm_name].copy()
        if len(group_df) == 0:
            raise ValueError(f"No Stage 07 rows found for {dataset_name} / {algorithm_name}")

        group_df = group_df.sort_values("test_f1", ascending=False).reset_index(drop=True)
        best_row = group_df.iloc[0].copy()

        model_file_name = str(best_row["model_file_name"])
        model_artifact_path = find_model_artifact_path(dataset_name, algorithm_name, model_file_name)
        patch_size = infer_patch_size_from_patch_set(best_row["patch_set"])

        anchor_rows.append({
            "dataset_name": dataset_name,
            "dataset_short": cfg["dataset_short"],
            "algorithm_name": algorithm_name,
            "anchor_model_file_name": model_file_name,
            "anchor_model_path": relative_path(model_artifact_path),
            "anchor_patch_set": best_row["patch_set"],
            "anchor_patch_size": patch_size,
            "anchor_feature_set": best_row["feature_set"],
            "stage07_test_rank": best_row.get("test_rank", np.nan),
            "stage07_test_f1": best_row["test_f1"],
            "stage07_test_precision": best_row.get("test_precision", np.nan),
            "stage07_test_recall": best_row.get("test_recall", np.nan),
            "stage07_test_specificity": best_row.get("test_specificity", np.nan),
        })

anchor_models_df = pd.DataFrame(anchor_rows)
save_csv(anchor_models_df, TABLES_DIR / "calibration_anchor_models.csv", overwrite=OVERWRITE_TABLES)
log_dataframe("Calibration Anchor Models", anchor_models_df)

log_output("[OK] Anchor models selected.")


## 10. Feature Çıkarma Yardımcıları

Bu bölümde Stage 03 ile uyumlu HOG, HSV, LBP ve PCA feature çıkarma fonksiyonları tanımlanır.


In [ ]:
HOG_CONFIGS = {
    "hog8": {
        "orientations": 9,
        "pixels_per_cell": (8, 8),
        "cells_per_block": (2, 2),
        "block_norm": "L2-Hys",
        "transform_sqrt": True,
    },
    "hog16": {
        "orientations": 9,
        "pixels_per_cell": (16, 16),
        "cells_per_block": (2, 2),
        "block_norm": "L2-Hys",
        "transform_sqrt": True,
    },
}

HSV_BINS = {
    "h": 16,
    "s": 8,
    "v": 8,
}

LBP_CONFIG = {
    "radius": 1,
    "points": 8,
    "method": "uniform",
}

pca_cache = {}

def extract_hog_features(patch_rgb, hog_key):
    gray = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2GRAY)
    cfg = HOG_CONFIGS[hog_key]
    features = hog(
        gray,
        orientations=cfg["orientations"],
        pixels_per_cell=cfg["pixels_per_cell"],
        cells_per_block=cfg["cells_per_block"],
        block_norm=cfg["block_norm"],
        transform_sqrt=cfg["transform_sqrt"],
        feature_vector=True,
        channel_axis=None,
    )
    return features.astype(np.float32)

def extract_hsv_features(patch_rgb):
    hsv = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2HSV)

    h_hist = cv2.calcHist([hsv], [0], None, [HSV_BINS["h"]], [0, 180]).flatten()
    s_hist = cv2.calcHist([hsv], [1], None, [HSV_BINS["s"]], [0, 256]).flatten()
    v_hist = cv2.calcHist([hsv], [2], None, [HSV_BINS["v"]], [0, 256]).flatten()

    def normalize_hist(hist):
        hist = hist.astype(np.float32)
        total = hist.sum()
        if total > 0:
            hist = hist / total
        return hist

    return np.concatenate([
        normalize_hist(h_hist),
        normalize_hist(s_hist),
        normalize_hist(v_hist),
    ]).astype(np.float32)

def extract_lbp_features(patch_rgb):
    gray = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2GRAY)
    radius = LBP_CONFIG["radius"]
    points = LBP_CONFIG["points"]
    method = LBP_CONFIG["method"]

    lbp = local_binary_pattern(gray, points, radius, method=method)
    n_bins = points + 2
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_bins + 1), range=(0, n_bins))
    hist = hist.astype(np.float32)
    total = hist.sum()
    if total > 0:
        hist = hist / total
    return hist.astype(np.float32)

def feature_set_uses_hog(feature_set):
    feature_set = str(feature_set)
    return feature_set.startswith("hog8") or feature_set.startswith("hog16")

def get_hog_key_from_feature_set(feature_set):
    feature_set = str(feature_set)
    if feature_set.startswith("hog8"):
        return "hog8"
    if feature_set.startswith("hog16"):
        return "hog16"
    return None

def feature_set_uses_pca(feature_set):
    return "_pca" in str(feature_set)

def load_pca_artifact(dataset_name, patch_set, hog_key):
    cache_key = (dataset_name, patch_set, hog_key)
    if cache_key in pca_cache:
        return pca_cache[cache_key]

    pca_artifact_path = (
        PROJECT_ROOT
        / "data"
        / "features"
        / dataset_name
        / "pca_artifacts"
        / f"{patch_set}__{hog_key}_pca.joblib"
    )

    if not pca_artifact_path.exists():
        raise FileNotFoundError(f"PCA artifact not found: {pca_artifact_path}")

    artifact = joblib.load(pca_artifact_path)

    if isinstance(artifact, dict):
        if "pca" not in artifact:
            raise KeyError(
                f"PCA artifact is a dict but does not contain key 'pca': {pca_artifact_path}. "
                f"Available keys: {list(artifact.keys())}"
            )
        pca = artifact["pca"]
    elif hasattr(artifact, "transform"):
        pca = artifact
    else:
        raise TypeError(
            f"Unsupported PCA artifact type at {pca_artifact_path}. "
            f"Expected dict with key 'pca' or sklearn-like transformer, got {type(artifact)}."
        )

    if not hasattr(pca, "transform"):
        raise TypeError(
            f"Loaded PCA object does not have transform(): {pca_artifact_path}. "
            f"Type: {type(pca)}"
        )

    pca_cache[cache_key] = pca
    return pca

def extract_feature_components(patch_rgb, dataset_name, patch_set, feature_set):
    feature_set = str(feature_set)
    hog_key = get_hog_key_from_feature_set(feature_set) if feature_set_uses_hog(feature_set) else None

    hog_features = None
    if hog_key is not None:
        hog_features = extract_hog_features(patch_rgb, hog_key)

    other_parts = []
    if "hsv" in feature_set:
        other_parts.append(extract_hsv_features(patch_rgb))
    if "lbp" in feature_set:
        other_parts.append(extract_lbp_features(patch_rgb))

    other_features = None
    if len(other_parts) > 0:
        other_features = np.concatenate(other_parts).astype(np.float32)

    return hog_key, hog_features, other_features

def build_feature_matrix_from_patches(patch_rgbs, dataset_name, patch_set, feature_set):
    feature_set = str(feature_set)

    hog_rows = []
    other_rows = []
    hog_key_seen = None

    for patch_rgb in patch_rgbs:
        hog_key, hog_features, other_features = extract_feature_components(
            patch_rgb=patch_rgb,
            dataset_name=dataset_name,
            patch_set=patch_set,
            feature_set=feature_set,
        )

        if hog_key is not None:
            hog_key_seen = hog_key
            hog_rows.append(hog_features)

        if other_features is not None:
            other_rows.append(other_features)

    feature_parts = []

    if len(hog_rows) > 0:
        hog_matrix = np.vstack(hog_rows).astype(np.float32)
        if feature_set_uses_pca(feature_set):
            pca = load_pca_artifact(dataset_name, patch_set, hog_key_seen)
            hog_matrix = pca.transform(hog_matrix).astype(np.float32)
        feature_parts.append(hog_matrix)

    if len(other_rows) > 0:
        other_matrix = np.vstack(other_rows).astype(np.float32)
        feature_parts.append(other_matrix)

    if len(feature_parts) == 0:
        raise ValueError(f"Unsupported feature_set: {feature_set}")

    X = np.hstack(feature_parts).astype(np.float32)
    return X

print("Feature extraction helpers ready.")


## 11. Model Yükleme ve Skorlama Yardımcıları

Bu bölümde joblib model dosyaları yüklenir ve sliding-window patch skorları hesaplanır.


In [ ]:
model_cache = {}

def unwrap_model_artifact(artifact):
    if isinstance(artifact, dict):
        for key in ["model", "pipeline", "estimator", "best_estimator", "clf", "classifier"]:
            if key in artifact:
                return artifact[key]
    return artifact

def load_model(model_path):
    model_path = Path(model_path)
    if model_path not in model_cache:
        artifact = joblib.load(model_path)
        model_cache[model_path] = unwrap_model_artifact(artifact)
    return model_cache[model_path]

def get_model_n_features(model):
    if hasattr(model, "n_features_in_"):
        return int(model.n_features_in_)
    if hasattr(model, "named_steps"):
        for step in model.named_steps.values():
            if hasattr(step, "n_features_in_"):
                return int(step.n_features_in_)
    return None

def score_feature_matrix(model, algorithm_name, X):
    if algorithm_name in ["linear_svm", "rbf_svm"]:
        if hasattr(model, "decision_function"):
            scores = model.decision_function(X)
            return np.asarray(scores).ravel().astype(float)
        if hasattr(model, "predict_proba"):
            scores = model.predict_proba(X)[:, 1]
            return np.asarray(scores).ravel().astype(float)
        preds = model.predict(X)
        return np.asarray(preds).ravel().astype(float)

    if algorithm_name == "random_forest":
        if hasattr(model, "predict_proba"):
            scores = model.predict_proba(X)[:, 1]
            return np.asarray(scores).ravel().astype(float)
        preds = model.predict(X)
        return np.asarray(preds).ravel().astype(float)

    raise ValueError(f"Unsupported algorithm_name: {algorithm_name}")

print("Model loading/scoring helpers ready.")


## 12. Sliding Window Yardımcıları

Bu bölümde full image üzerinde aday patch pencereleri üretilir ve model skorları çıkarılır.


In [ ]:
def generate_sliding_windows(image_rgb, patch_size, stride):
    img_h, img_w = image_rgb.shape[:2]

    if img_w < patch_size or img_h < patch_size:
        return []

    x_positions = list(range(0, max(1, img_w - patch_size + 1), stride))
    y_positions = list(range(0, max(1, img_h - patch_size + 1), stride))

    last_x = img_w - patch_size
    last_y = img_h - patch_size

    if last_x >= 0 and last_x not in x_positions:
        x_positions.append(last_x)
    if last_y >= 0 and last_y not in y_positions:
        y_positions.append(last_y)

    windows = []
    for y in y_positions:
        for x in x_positions:
            windows.append({
                "x1": int(x),
                "y1": int(y),
                "x2": int(x + patch_size),
                "y2": int(y + patch_size),
            })
    return windows

def load_image_rgb(image_path):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return image_rgb

def compute_raw_window_scores_for_image(
    image_row,
    model,
    algorithm_name,
    dataset_name,
    patch_set,
    patch_size,
    feature_set,
    stride_divisor,
):
    image_path = image_row["image_path"]
    image_rgb = load_image_rgb(image_path)

    if IMAGE_SCALE != 1.0:
        raise ValueError("This notebook currently supports IMAGE_SCALE = 1.0 only.")

    stride = max(4, int(patch_size // stride_divisor))
    windows = generate_sliding_windows(image_rgb, patch_size, stride)

    if len(windows) == 0:
        return pd.DataFrame()

    patch_rgbs = []
    metadata_rows = []

    for window_index, window in enumerate(windows):
        patch_rgb = image_rgb[window["y1"]:window["y2"], window["x1"]:window["x2"]].copy()
        patch_rgbs.append(patch_rgb)

        metadata_rows.append({
            "dataset_name": dataset_name,
            "image_id": image_row["image_id"],
            "image_name": image_row["image_name"],
            "image_path": image_path,
            "image_width": int(image_row["image_width"]),
            "image_height": int(image_row["image_height"]),
            "varroa_count": int(image_row["varroa_count"]),
            "gt_boxes_json": image_row["gt_boxes_json"],
            "window_index": int(window_index),
            "x1": int(window["x1"]),
            "y1": int(window["y1"]),
            "x2": int(window["x2"]),
            "y2": int(window["y2"]),
            "patch_size": int(patch_size),
            "stride_divisor": int(stride_divisor),
            "stride": int(stride),
            "feature_set": feature_set,
        })

    X = build_feature_matrix_from_patches(
        patch_rgbs=patch_rgbs,
        dataset_name=dataset_name,
        patch_set=patch_set,
        feature_set=feature_set,
    )

    expected_features = get_model_n_features(model)
    if expected_features is not None and X.shape[1] != expected_features:
        raise ValueError(
            f"Feature dimension mismatch for {dataset_name}/{algorithm_name}/{feature_set}: "
            f"extracted={X.shape[1]}, model_expected={expected_features}"
        )

    scores = score_feature_matrix(model, algorithm_name, X)
    raw_scores_df = pd.DataFrame(metadata_rows)
    raw_scores_df["score"] = scores

    return raw_scores_df

def compute_raw_scores_for_anchor_and_stride(anchor_row, stride_divisor, subset_df):
    dataset_name = anchor_row["dataset_name"]
    algorithm_name = anchor_row["algorithm_name"]
    model_path = anchor_row["anchor_model_path"]
    patch_set = anchor_row["anchor_patch_set"]
    patch_size = int(anchor_row["anchor_patch_size"])
    feature_set = anchor_row["anchor_feature_set"]

    model = load_model(model_path)

    raw_parts = []
    start_time = time.time()

    for idx, image_row in subset_df.reset_index(drop=True).iterrows():
        if (idx + 1) % 10 == 0 or idx == 0:
            print(
                f"  {dataset_name} / {algorithm_name} / stride_divisor={stride_divisor}: "
                f"image {idx + 1}/{len(subset_df)}"
            )

        raw_df = compute_raw_window_scores_for_image(
            image_row=image_row,
            model=model,
            algorithm_name=algorithm_name,
            dataset_name=dataset_name,
            patch_set=patch_set,
            patch_size=patch_size,
            feature_set=feature_set,
            stride_divisor=stride_divisor,
        )
        if len(raw_df) > 0:
            raw_parts.append(raw_df)

    if len(raw_parts) == 0:
        combined_df = pd.DataFrame()
    else:
        combined_df = pd.concat(raw_parts, ignore_index=True)

    elapsed = time.time() - start_time
    combined_df.attrs["raw_score_runtime_seconds"] = elapsed

    return combined_df

print("Sliding-window helpers ready.")


## 13. Eşleştirme ve Kümeleme Yardımcıları

Bu bölümde IoU, weighted-center kümeleme ve ground-truth eşleştirme fonksiyonları tanımlanır.


In [ ]:
def compute_iou(box_a, box_b):
    if isinstance(box_a, dict):
        ax1, ay1, ax2, ay2 = box_a["x1"], box_a["y1"], box_a["x2"], box_a["y2"]
    else:
        ax1, ay1, ax2, ay2 = box_a

    if isinstance(box_b, dict):
        bx1, by1, bx2, by2 = box_b["x1"], box_b["y1"], box_b["x2"], box_b["y2"]
    else:
        bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union_area = area_a + area_b - inter_area

    if union_area <= 0:
        return 0.0
    return float(inter_area / union_area)

def clip_box_to_image(box, image_width, image_height):
    clipped = box.copy()
    clipped["x1"] = float(max(0, min(image_width, clipped["x1"])))
    clipped["y1"] = float(max(0, min(image_height, clipped["y1"])))
    clipped["x2"] = float(max(0, min(image_width, clipped["x2"])))
    clipped["y2"] = float(max(0, min(image_height, clipped["y2"])))

    if clipped["x2"] < clipped["x1"]:
        clipped["x1"], clipped["x2"] = clipped["x2"], clipped["x1"]
    if clipped["y2"] < clipped["y1"]:
        clipped["y1"], clipped["y2"] = clipped["y2"], clipped["y1"]

    return clipped

def build_candidate_detections_from_raw_scores(image_raw_scores_df, score_threshold, max_candidates_per_image=MAX_CANDIDATES_PER_IMAGE):
    candidate_df = image_raw_scores_df[image_raw_scores_df["score"] >= score_threshold].copy()

    if max_candidates_per_image is not None and len(candidate_df) > max_candidates_per_image:
        candidate_df = candidate_df.sort_values("score", ascending=False).head(max_candidates_per_image).copy()

    detections = []
    for _, row in candidate_df.iterrows():
        detections.append({
            "x1": float(row["x1"]),
            "y1": float(row["y1"]),
            "x2": float(row["x2"]),
            "y2": float(row["y2"]),
            "score": float(row["score"]),
            "window_index": int(row["window_index"]),
        })
    return detections

def apply_weighted_center_clustering(
    detections,
    image_width,
    image_height,
    patch_size,
    score_threshold,
    cluster_iou_threshold=0.30,
    weighted_box_size_factor=1.25,
):
    if len(detections) == 0:
        return []

    remaining = sorted(detections, key=lambda d: d["score"], reverse=True)
    final_detections = []

    while len(remaining) > 0:
        seed = remaining.pop(0)
        cluster = [seed]
        new_remaining = []

        for det in remaining:
            if compute_iou(seed, det) >= cluster_iou_threshold:
                cluster.append(det)
            else:
                new_remaining.append(det)

        remaining = new_remaining

        centers_x = np.array([(d["x1"] + d["x2"]) / 2 for d in cluster], dtype=float)
        centers_y = np.array([(d["y1"] + d["y2"]) / 2 for d in cluster], dtype=float)
        scores = np.array([d["score"] for d in cluster], dtype=float)

        weights = np.maximum(scores - float(score_threshold), 0.0) + 1e-6
        weighted_cx = float(np.sum(centers_x * weights) / np.sum(weights))
        weighted_cy = float(np.sum(centers_y * weights) / np.sum(weights))

        final_size = float(patch_size) * float(weighted_box_size_factor)
        half_size = final_size / 2.0

        final_box = {
            "x1": weighted_cx - half_size,
            "y1": weighted_cy - half_size,
            "x2": weighted_cx + half_size,
            "y2": weighted_cy + half_size,
            "score": float(np.max(scores)),
            "cluster_size": int(len(cluster)),
            "cluster_mean_score": float(np.mean(scores)),
            "weighted_cx": weighted_cx,
            "weighted_cy": weighted_cy,
        }

        final_box = clip_box_to_image(final_box, image_width, image_height)
        final_detections.append(final_box)

    final_detections = sorted(final_detections, key=lambda d: d["score"], reverse=True)
    return final_detections

def match_detections_to_gt(detections, gt_boxes, iou_threshold):
    # Safety filter: GT matching must only use Varroa boxes.
    gt_boxes = [box for box in gt_boxes if box.get("is_varroa", True)]

    matched_gt_indices = set()
    tp = 0
    fp = 0
    matched_ious = []

    detections_sorted = sorted(detections, key=lambda d: d["score"], reverse=True)

    for det in detections_sorted:
        best_iou = 0.0
        best_gt_idx = None

        for gt_idx, gt_box in enumerate(gt_boxes):
            if gt_idx in matched_gt_indices:
                continue
            iou = compute_iou(det, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_gt_idx is not None and best_iou >= iou_threshold:
            tp += 1
            matched_gt_indices.add(best_gt_idx)
            matched_ious.append(best_iou)
        else:
            fp += 1

    fn = max(0, len(gt_boxes) - len(matched_gt_indices))
    return tp, fp, fn, matched_ious

def safe_divide(numerator, denominator):
    if denominator == 0:
        return 0.0
    return float(numerator / denominator)

print("Post-processing and matching helpers ready.")


## 14. Kalibrasyon Grid'i

Bu bölümde her algoritma için test edilecek threshold, stride ve kümeleme parametre kombinasyonları oluşturulur.


In [ ]:
def build_algorithm_param_grid(algorithm_name, param_grid):
    rows = []
    keys = list(param_grid.keys())
    threshold_column = THRESHOLD_COLUMN_BY_ALGORITHM[algorithm_name]

    for values in product(*[param_grid[key] for key in keys]):
        row = dict(zip(keys, values))
        row["algorithm_name"] = algorithm_name
        row["threshold_column"] = threshold_column
        row["threshold_value"] = row[threshold_column]
        rows.append(row)

    grid_df = pd.DataFrame(rows)
    grid_df = grid_df.loc[:, ~grid_df.columns.duplicated()].copy()
    return grid_df

grid_parts = []
for algorithm_name, param_grid in PARAM_GRIDS.items():
    algorithm_grid_df = build_algorithm_param_grid(algorithm_name, param_grid)
    duplicated_cols = algorithm_grid_df.columns[algorithm_grid_df.columns.duplicated()].tolist()
    if len(duplicated_cols) > 0:
        raise ValueError(f"Duplicate columns in grid for {algorithm_name}: {duplicated_cols}")
    grid_parts.append(algorithm_grid_df)

all_param_grid_df = pd.concat(grid_parts, ignore_index=True)
all_param_grid_df["param_combo_index"] = np.arange(1, len(all_param_grid_df) + 1)

front_cols = [
    "param_combo_index",
    "algorithm_name",
    "threshold_column",
    "threshold_value",
    "score_threshold",
    "decision_function_score_threshold",
    "predict_proba_score_threshold",
    "stride_divisor",
    "cluster_iou_threshold",
    "weighted_box_size_factor",
]

existing_front_cols = [col for col in front_cols if col in all_param_grid_df.columns]
other_cols = [col for col in all_param_grid_df.columns if col not in existing_front_cols]
all_param_grid_df = all_param_grid_df[existing_front_cols + other_cols].copy()

save_csv(
    all_param_grid_df,
    TABLES_DIR / "preliminary_calibration_grid.csv",
    overwrite=OVERWRITE_TABLES,
)

log_dataframe("Preliminary Calibration Grid", all_param_grid_df, max_rows=30)

combo_counts_df = (
    all_param_grid_df
    .groupby("algorithm_name")
    .size()
    .reset_index(name="combination_count")
)

log_dataframe("Calibration Combination Count by Algorithm", combo_counts_df)

threshold_check_df = (
    all_param_grid_df
    .groupby("algorithm_name")["threshold_value"]
    .agg(lambda s: sorted(set([float(x) for x in s])))
    .reset_index(name="threshold_values")
)
log_dataframe("Threshold Value Check", threshold_check_df)


## 15. Kalibrasyon Değerlendirme Fonksiyonu

Bu bölümde tek bir anchor model ve parametre kombinasyonu için detection metrikleri hesaplanır.


In [ ]:
def evaluate_param_combination_from_raw_scores(
    anchor_row,
    raw_scores_df,
    param_row,
):
    start_time = time.time()

    dataset_name = anchor_row["dataset_name"]
    algorithm_name = anchor_row["algorithm_name"]
    patch_size = int(anchor_row["anchor_patch_size"])

    threshold_column = param_row["threshold_column"]
    score_threshold = float(param_row["threshold_value"])
    stride_divisor = int(param_row["stride_divisor"])
    expected_stride = max(4, int(patch_size // stride_divisor))

    cluster_iou_threshold = float(param_row["cluster_iou_threshold"])
    weighted_box_size_factor = float(param_row["weighted_box_size_factor"])

    if len(raw_scores_df) == 0:
        raise ValueError("raw_scores_df is empty.")

    raw_stride_values = sorted(raw_scores_df["stride"].dropna().unique().tolist())
    if len(raw_stride_values) != 1 or int(raw_stride_values[0]) != expected_stride:
        raise ValueError(
            f"Stride mismatch for {dataset_name}/{algorithm_name}: "
            f"expected={expected_stride}, raw_stride_values={raw_stride_values}"
        )

    stride = expected_stride

    metric_accumulator = {}
    image_level_rows = []

    for matching_iou in MATCHING_IOU_THRESHOLDS:
        metric_accumulator[matching_iou] = {
            "tp": 0,
            "fp": 0,
            "fn": 0,
            "matched_ious": [],
            "negative_image_count": 0,
            "negative_total_fp": 0,
            "negative_correct_count": 0,
        }

    total_window_count = int(len(raw_scores_df))
    total_candidate_count = 0
    total_final_detection_count = 0
    image_count = int(raw_scores_df["image_name"].nunique())

    for image_name, image_raw_scores_df in raw_scores_df.groupby("image_name"):
        if image_raw_scores_df["gt_boxes_json"].nunique() != 1:
            raise ValueError(f"gt_boxes_json inconsistency for image {image_name}")

        first_row = image_raw_scores_df.iloc[0]
        image_width = int(first_row["image_width"])
        image_height = int(first_row["image_height"])
        varroa_count = int(first_row["varroa_count"])
        gt_boxes = json.loads(first_row["gt_boxes_json"])
        gt_boxes = [box for box in gt_boxes if box.get("is_varroa", True)]

        if len(gt_boxes) != varroa_count:
            raise ValueError(
                f"GT box count mismatch for image {image_name}: "
                f"varroa_count={varroa_count}, len(gt_boxes)={len(gt_boxes)}"
            )

        candidates = build_candidate_detections_from_raw_scores(
            image_raw_scores_df=image_raw_scores_df,
            score_threshold=score_threshold,
            max_candidates_per_image=MAX_CANDIDATES_PER_IMAGE,
        )
        final_detections = apply_weighted_center_clustering(
            detections=candidates,
            image_width=image_width,
            image_height=image_height,
            patch_size=patch_size,
            score_threshold=score_threshold,
            cluster_iou_threshold=cluster_iou_threshold,
            weighted_box_size_factor=weighted_box_size_factor,
        )

        total_candidate_count += len(candidates)
        total_final_detection_count += len(final_detections)

        image_row_base = {
            "dataset_name": dataset_name,
            "algorithm_name": algorithm_name,
            "anchor_model_file_name": anchor_row["anchor_model_file_name"],
            "image_name": image_name,
            "varroa_count": varroa_count,
            "window_count": len(image_raw_scores_df),
            "candidate_count": len(candidates),
            "final_detection_count": len(final_detections),
            "threshold_column": threshold_column,
            "threshold_value": score_threshold,
            "score_threshold": score_threshold,
            "stride_divisor": stride_divisor,
            "stride": stride,
            "cluster_iou_threshold": cluster_iou_threshold,
            "weighted_box_size_factor": weighted_box_size_factor,
        }

        for matching_iou in MATCHING_IOU_THRESHOLDS:
            tp, fp, fn, matched_ious = match_detections_to_gt(
                detections=final_detections,
                gt_boxes=gt_boxes,
                iou_threshold=matching_iou,
            )

            acc = metric_accumulator[matching_iou]
            acc["tp"] += tp
            acc["fp"] += fp
            acc["fn"] += fn
            acc["matched_ious"].extend(matched_ious)

            if varroa_count == 0:
                acc["negative_image_count"] += 1
                acc["negative_total_fp"] += fp
                if fp == 0:
                    acc["negative_correct_count"] += 1

            prefix = f"iou_{str(matching_iou).replace('.', '_')}"
            image_row_base[f"tp_{prefix}"] = tp
            image_row_base[f"fp_{prefix}"] = fp
            image_row_base[f"fn_{prefix}"] = fn
            image_row_base[f"precision_{prefix}"] = safe_divide(tp, tp + fp)
            image_row_base[f"recall_{prefix}"] = safe_divide(tp, tp + fn)
            image_row_base[f"f1_{prefix}"] = safe_divide(2 * tp, 2 * tp + fp + fn)

        image_level_rows.append(image_row_base)

    runtime_seconds = time.time() - start_time

    summary = {
        "dataset_name": dataset_name,
        "algorithm_name": algorithm_name,
        "anchor_model_file_name": anchor_row["anchor_model_file_name"],
        "anchor_patch_set": anchor_row["anchor_patch_set"],
        "anchor_patch_size": patch_size,
        "anchor_feature_set": anchor_row["anchor_feature_set"],
        "stage07_test_f1": anchor_row["stage07_test_f1"],
        "param_combo_index": int(param_row["param_combo_index"]),
        "threshold_column": threshold_column,
        "threshold_value": score_threshold,
        "score_threshold": score_threshold,
        "stride_divisor": stride_divisor,
        "stride": stride,
        "cluster_iou_threshold": cluster_iou_threshold,
        "weighted_box_size_factor": weighted_box_size_factor,
        "image_count": image_count,
        "total_window_count": total_window_count,
        "total_candidate_count": total_candidate_count,
        "total_final_detection_count": total_final_detection_count,
        "mean_candidates_per_image": safe_divide(total_candidate_count, image_count),
        "mean_final_detections_per_image": safe_divide(total_final_detection_count, image_count),
        "raw_score_runtime_seconds": float(raw_scores_df.attrs.get("raw_score_runtime_seconds", np.nan)),
        "evaluation_runtime_seconds": runtime_seconds,
    }

    for matching_iou, acc in metric_accumulator.items():
        suffix = str(matching_iou).replace(".", "_")
        tp = acc["tp"]
        fp = acc["fp"]
        fn = acc["fn"]
        precision = safe_divide(tp, tp + fp)
        recall = safe_divide(tp, tp + fn)
        f1 = safe_divide(2 * tp, 2 * tp + fp + fn)
        fp_per_image = safe_divide(fp, image_count)
        fn_per_image = safe_divide(fn, image_count)
        negative_fp_per_negative_image = safe_divide(acc["negative_total_fp"], acc["negative_image_count"])
        negative_accuracy = safe_divide(acc["negative_correct_count"], acc["negative_image_count"])

        summary[f"tp_iou_{suffix}"] = tp
        summary[f"fp_iou_{suffix}"] = fp
        summary[f"fn_iou_{suffix}"] = fn
        summary[f"detection_precision_iou_{suffix}"] = precision
        summary[f"detection_recall_iou_{suffix}"] = recall
        summary[f"detection_f1_iou_{suffix}"] = f1
        summary[f"false_positives_per_image_iou_{suffix}"] = fp_per_image
        summary[f"false_negatives_per_image_iou_{suffix}"] = fn_per_image
        summary[f"negative_image_count_iou_{suffix}"] = acc["negative_image_count"]
        summary[f"negative_total_fp_iou_{suffix}"] = acc["negative_total_fp"]
        summary[f"negative_fp_per_negative_image_iou_{suffix}"] = negative_fp_per_negative_image
        summary[f"negative_accuracy_iou_{suffix}"] = negative_accuracy
        summary[f"mean_matched_iou_iou_{suffix}"] = float(np.mean(acc["matched_ious"])) if len(acc["matched_ious"]) > 0 else 0.0

    image_level_df = pd.DataFrame(image_level_rows)
    return summary, image_level_df

print("Calibration evaluation helper ready.")


## 16. Ön Kalibrasyon

Bu bölümde anchor modeller valid alt küme üzerinde çalıştırılır ve parametre kombinasyonları karşılaştırılır.


In [ ]:
if not RUN_PRELIMINARY_CALIBRATION:
    raise RuntimeError("RUN_PRELIMINARY_CALIBRATION is False. Nothing to run.")

raw_score_cache = {}
summary_rows = []
image_level_result_parts = []

calibration_start_time = time.time()

for anchor_idx, anchor_row in anchor_models_df.iterrows():
    dataset_name = anchor_row["dataset_name"]
    algorithm_name = anchor_row["algorithm_name"]
    subset_df = valid_subset_by_dataset[dataset_name].reset_index(drop=True)

    log_output("=" * 100)
    log_output(
        f"Starting calibration group: {dataset_name} / {algorithm_name} | "
        f"anchor={anchor_row['anchor_model_file_name']} | images={len(subset_df)}"
    )

    algorithm_grid_df = all_param_grid_df[
        all_param_grid_df["algorithm_name"] == algorithm_name
    ].copy().reset_index(drop=True)

    stride_divisors = sorted(algorithm_grid_df["stride_divisor"].unique())

    for stride_divisor in stride_divisors:
        cache_key = (dataset_name, algorithm_name, int(stride_divisor))
        log_output(f"[RAW SCORE] Computing raw scores for {cache_key}")

        raw_scores_df = compute_raw_scores_for_anchor_and_stride(
            anchor_row=anchor_row,
            stride_divisor=int(stride_divisor),
            subset_df=subset_df,
        )
        raw_score_cache[cache_key] = raw_scores_df

        log_output(
            f"[RAW SCORE DONE] {cache_key} | rows={len(raw_scores_df)} | "
            f"runtime={raw_scores_df.attrs.get('raw_score_runtime_seconds', np.nan):.2f}s"
        )

    group_start = time.time()

    for combo_idx, param_row in algorithm_grid_df.iterrows():
        stride_divisor = int(param_row["stride_divisor"])
        cache_key = (dataset_name, algorithm_name, stride_divisor)
        raw_scores_df = raw_score_cache[cache_key]

        if (combo_idx + 1) % 8 == 0 or combo_idx == 0:
            log_output(
                f"Evaluating {dataset_name}/{algorithm_name}: "
                f"combo {combo_idx + 1}/{len(algorithm_grid_df)}"
            )

        summary, image_level_df = evaluate_param_combination_from_raw_scores(
            anchor_row=anchor_row,
            raw_scores_df=raw_scores_df,
            param_row=param_row,
        )
        summary_rows.append(summary)
        image_level_result_parts.append(image_level_df)

    group_elapsed = time.time() - group_start
    log_output(f"[GROUP DONE] {dataset_name} / {algorithm_name} | sweep_runtime={group_elapsed:.2f}s")

total_calibration_runtime = time.time() - calibration_start_time

preliminary_results_df = pd.DataFrame(summary_rows)
preliminary_image_results_df = pd.concat(image_level_result_parts, ignore_index=True)

save_csv(
    preliminary_results_df,
    TABLES_DIR / "preliminary_calibration_results.csv",
    overwrite=OVERWRITE_TABLES,
    note="Group-level preliminary detection calibration results.",
)

save_csv(
    preliminary_image_results_df,
    TABLES_DIR / "preliminary_calibration_image_level_results.csv",
    overwrite=OVERWRITE_TABLES,
    note="Image-level preliminary detection calibration results.",
)

log_dataframe("Preliminary Calibration Results Preview", preliminary_results_df, max_rows=20)
log_output(f"[OK] Preliminary calibration completed in {total_calibration_runtime:.2f} seconds.")

threshold_result_check_df = (
    preliminary_image_results_df
    .groupby(["dataset_name", "algorithm_name"])["score_threshold"]
    .agg(lambda s: sorted(set([float(x) for x in s])))
    .reset_index(name="used_score_thresholds")
)
log_dataframe("Used Score Thresholds by Group", threshold_result_check_df)


## 17. Parametre Seçimi

Bu bölümde kalibrasyon sonuçları sıralanır ve her veri seti-algoritma grubu için kullanılacak detection parametreleri seçilir.


In [ ]:
primary_suffix = str(PRIMARY_MATCHING_IOU).replace(".", "_")
rank_columns = [
    f"detection_f1_iou_{primary_suffix}",
    f"false_positives_per_image_iou_{primary_suffix}",
    f"detection_recall_iou_{primary_suffix}",
    "mean_final_detections_per_image",
    "evaluation_runtime_seconds",
]
ascending_values = [
    False,
    True,
    False,
    True,
    True,
]

ranked_parts = []
selected_rows = []

for (dataset_name, algorithm_name), group_df in preliminary_results_df.groupby(["dataset_name", "algorithm_name"]):
    group_ranked = group_df.sort_values(
        by=rank_columns,
        ascending=ascending_values,
    ).reset_index(drop=True)

    group_ranked["group_rank"] = np.arange(1, len(group_ranked) + 1)
    ranked_parts.append(group_ranked)

    selected = group_ranked.iloc[0].copy()
    selected["selection_reason"] = (
        f"Selected by highest detection_f1_iou_{primary_suffix}, "
        "then lower FP/image, higher recall, lower final detections/image, lower runtime."
    )
    selected_rows.append(selected)

ranked_calibration_results_df = pd.concat(ranked_parts, ignore_index=True)
selected_detection_parameters_df = pd.DataFrame(selected_rows)

selected_front_cols = [
    "dataset_name",
    "algorithm_name",
    "anchor_model_file_name",
    "anchor_patch_set",
    "anchor_patch_size",
    "anchor_feature_set",
    "stage07_test_f1",
    "threshold_column",
    "threshold_value",
    "score_threshold",
    "stride_divisor",
    "stride",
    "cluster_iou_threshold",
    "weighted_box_size_factor",
    f"detection_f1_iou_{primary_suffix}",
    f"detection_precision_iou_{primary_suffix}",
    f"detection_recall_iou_{primary_suffix}",
    f"false_positives_per_image_iou_{primary_suffix}",
    f"tp_iou_{primary_suffix}",
    f"fp_iou_{primary_suffix}",
    f"fn_iou_{primary_suffix}",
    "mean_candidates_per_image",
    "mean_final_detections_per_image",
    "raw_score_runtime_seconds",
    "evaluation_runtime_seconds",
    "selection_reason",
]
selected_existing_front_cols = [col for col in selected_front_cols if col in selected_detection_parameters_df.columns]
selected_other_cols = [col for col in selected_detection_parameters_df.columns if col not in selected_existing_front_cols]
selected_detection_parameters_df = selected_detection_parameters_df[selected_existing_front_cols + selected_other_cols]

save_csv(
    ranked_calibration_results_df,
    TABLES_DIR / "ranked_preliminary_calibration_results.csv",
    overwrite=OVERWRITE_TABLES,
    note="Ranked preliminary calibration results for all groups and parameter combinations.",
)

save_csv(
    selected_detection_parameters_df,
    TABLES_DIR / "selected_detection_parameters.csv",
    overwrite=OVERWRITE_TABLES,
    note="Selected preliminary detection parameters per dataset × algorithm group.",
)

log_dataframe("Selected Detection Parameters", selected_detection_parameters_df, max_rows=10)


## 18. Sınır Kontrolü

Bu bölümde seçilen parametrelerin grid sınırında kalıp kalmadığı kontrol edilir.


In [ ]:
boundary_rows = []

for _, row in selected_detection_parameters_df.iterrows():
    dataset_name = row["dataset_name"]
    algorithm_name = row["algorithm_name"]
    threshold_column = THRESHOLD_COLUMN_BY_ALGORITHM[algorithm_name]
    grid = PARAM_GRIDS[algorithm_name]

    tested_threshold_values = grid[threshold_column]
    tested_stride_divisors = grid["stride_divisor"]

    selected_threshold = float(row["score_threshold"])
    selected_stride_divisor = int(row["stride_divisor"])

    threshold_boundary = "inside_tested_range"
    if selected_threshold == min(tested_threshold_values):
        threshold_boundary = "lower_boundary"
    elif selected_threshold == max(tested_threshold_values):
        threshold_boundary = "upper_boundary"

    stride_boundary = "inside_tested_range"
    if selected_stride_divisor == min(tested_stride_divisors):
        stride_boundary = "lower_divisor_boundary_less_dense"
    elif selected_stride_divisor == max(tested_stride_divisors):
        stride_boundary = "upper_divisor_boundary_more_dense"

    boundary_rows.append({
        "dataset_name": dataset_name,
        "algorithm_name": algorithm_name,
        "threshold_column": threshold_column,
        "tested_threshold_values": json.dumps(tested_threshold_values),
        "selected_threshold": selected_threshold,
        "threshold_boundary": threshold_boundary,
        "tested_stride_divisors": json.dumps(tested_stride_divisors),
        "selected_stride_divisor": selected_stride_divisor,
        "stride_boundary": stride_boundary,
        "note": "Boundary values are not automatically changed in this notebook.",
    })

boundary_check_df = pd.DataFrame(boundary_rows)
save_csv(boundary_check_df, TABLES_DIR / "parameter_boundary_check.csv", overwrite=OVERWRITE_TABLES)
log_dataframe("Parameter Boundary Check", boundary_check_df)


## 19. Kalite Kontrolü

Bu bölümde kalibrasyon sonuçlarında aday üretmeyen veya sıfır F1 veren gruplar işaretlenir.


In [ ]:
quality_rows = []

for (dataset_name, algorithm_name), group_df in preliminary_results_df.groupby(["dataset_name", "algorithm_name"]):
    threshold_values = sorted(set([float(x) for x in group_df["score_threshold"].dropna()]))
    nonzero_candidate_runs = int((group_df["total_candidate_count"] > 0).sum())
    nonzero_detection_runs = int((group_df["total_final_detection_count"] > 0).sum())
    best_f1 = float(group_df[f"detection_f1_iou_{primary_suffix}"].max())
    group_status = "OK"
    notes = []

    if len(threshold_values) == 0:
        group_status = "WARNING"
        notes.append("No threshold values found.")
    if nonzero_candidate_runs == 0:
        group_status = "WARNING"
        notes.append("All parameter combinations produced zero candidates.")
    if nonzero_detection_runs == 0:
        group_status = "WARNING"
        notes.append("All parameter combinations produced zero final detections.")
    if best_f1 == 0:
        group_status = "WARNING"
        notes.append("Best detection F1 is zero.")

    quality_rows.append({
        "dataset_name": dataset_name,
        "algorithm_name": algorithm_name,
        "tested_score_thresholds": json.dumps(threshold_values),
        "combination_count": int(len(group_df)),
        "nonzero_candidate_runs": nonzero_candidate_runs,
        "nonzero_detection_runs": nonzero_detection_runs,
        f"best_detection_f1_iou_{primary_suffix}": best_f1,
        "quality_status": group_status,
        "notes": " | ".join(notes),
    })

quality_check_df = pd.DataFrame(quality_rows)
save_csv(quality_check_df, TABLES_DIR / "calibration_quality_check.csv", overwrite=OVERWRITE_TABLES)
log_dataframe("Calibration Quality Check", quality_check_df)


## 20. Görseller

Bu bölümde seçilen parametrelerin detection F1 ve false-positive davranışı özetlenir.


In [ ]:
plot_df = selected_detection_parameters_df.copy()
plot_df["group_label"] = plot_df["dataset_name"] + " / " + plot_df["algorithm_name"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(plot_df["group_label"], plot_df[f"detection_f1_iou_{primary_suffix}"])
ax.set_title(f"Selected Preliminary Detection F1 by Group — IoU@{PRIMARY_MATCHING_IOU}")
ax.set_xlabel("Dataset / Algorithm")
ax.set_ylabel("Detection F1")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig_path = FIGURES_DIR / "best_detection_f1_by_group.png"
save_current_figure(fig_path, "Detection calibration figure", overwrite=OVERWRITE_FIGURES)
plt.show()
# Figure path logged by save_current_figure.

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(plot_df["group_label"], plot_df[f"false_positives_per_image_iou_{primary_suffix}"])
ax.set_title(f"Selected False Positives per Image by Group — IoU@{PRIMARY_MATCHING_IOU}")
ax.set_xlabel("Dataset / Algorithm")
ax.set_ylabel("False positives per image")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig_path = FIGURES_DIR / "best_fp_per_image_by_group.png"
save_current_figure(fig_path, "Detection calibration figure", overwrite=OVERWRITE_FIGURES)
plt.show()
# Figure path logged by save_current_figure.


## 21. Final Durum

Bu bölümde notebook çıktılarının kısa durum özeti kaydedilir.


In [ ]:
overall_status = "READY_FOR_NEXT_STAGE"
warning_count = int((quality_check_df["quality_status"] != "OK").sum())
if warning_count > 0:
    overall_status = "COMPLETED_WITH_WARNINGS"

status_rows = []

for dataset_name, subset_df in valid_subset_by_dataset.items():
    status_rows.append({
        "item": f"{dataset_name}_valid_subset_image_count",
        "value": len(subset_df),
    })

status_rows.extend([
    {"item": "anchor_model_count", "value": len(anchor_models_df)},
    {"item": "calibration_group_count", "value": selected_detection_parameters_df[["dataset_name", "algorithm_name"]].drop_duplicates().shape[0]},
    {"item": "total_calibration_combinations_evaluated", "value": len(preliminary_results_df)},
    {"item": "matching_iou_thresholds", "value": json.dumps(MATCHING_IOU_THRESHOLDS)},
    {"item": "primary_matching_iou", "value": PRIMARY_MATCHING_IOU},
    {"item": "postprocess_method", "value": POSTPROCESS_METHOD},
    {"item": "image_scale", "value": IMAGE_SCALE},
    {"item": "max_candidates_per_image", "value": MAX_CANDIDATES_PER_IMAGE},
    {"item": "pca_cache_entry_count", "value": len(pca_cache)},
    {"item": "calibration_quality_warning_count", "value": warning_count},
    {"item": "total_calibration_runtime_seconds", "value": round(total_calibration_runtime, 3)},
    {"item": "status", "value": overall_status},
])

notebook_status_df = pd.DataFrame(status_rows)
save_csv(notebook_status_df, TABLES_DIR / "notebook_status.csv", overwrite=OVERWRITE_TABLES)
log_dataframe("Notebook Status", notebook_status_df, max_rows=50)

log_output(f"[STATUS] 00_preliminary_detection_parameter_calibration: {overall_status}")
